# Phase 2: Hybrid Weapon Detection Training — Version 7 (Optimized)

This version incorporates optimizations based on early training logs from v6:
1. **Cosine Annealing Scheduler**: For smoother convergence and better global minimum search.
2. **Differential Learning Rate**: Dropping LR during Phase 2 (unfreeze) to preserve pretrained features.
3. **Enhanced Augmentations**: Increased probability for CCTV-like noise (Blur, ToGray).
4. **Sanitized Data Handling**: Assumes a cleaned dataset (duplicates/segments removed).

| Component | Location | Speed |
|-----------|----------|-------|
| Source Code & Model Weights | Google Drive (persistent) | — |
| Dataset (41k images, 10 GB) | `yolo_dataset.zip` on GDrive → extracted to Colab SSD | ⚡ ~100 MB/s |
| Training I/O | Colab local SSD (`/content/`) | ⚡ Native |
| Checkpoints | Google Drive `models/weights/` | Auto-saved |


## Step 1 — Environment & Dependencies
Install core packages and fix the **Numpy < 2.0** binary incompatibility.

In [ ]:
# ── Install packages ────────────────────────────────────────────────────────
%pip install -q ultralytics albumentations timm

# ── Force Numpy < 2.0 ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.0"], check=True)

import numpy as np, torch, os, sys, time, shutil, yaml
from pathlib import Path
from google.colab import drive

print(f"✅ Numpy  : {np.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU    : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU! Go to Runtime > Change runtime type > T4 GPU")

## Step 2 — Configuration & Dataset Setup

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              USER-EDITABLE CONFIGURATION — change these paths            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

ZIP_GDRIVE_PATH = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/yolo_dataset.zip"
GDRIVE_PROJECT_PATH = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/"
LOCAL_DATASET_DIR = "/content/yolo_dataset"

# ── 1. Mount Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── 2. Align project root ─────────────────────────────────────────────────
PROJECT_ROOT = Path(GDRIVE_PROJECT_PATH)
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print(f"✅ Project root: {PROJECT_ROOT}")

# ── 3. Extract Dataset ────────────────────────────────────────────────────
DATA_YAML_PATH = Path(LOCAL_DATASET_DIR) / "data.yaml"
if not DATA_YAML_PATH.exists():
    if not os.path.exists(ZIP_GDRIVE_PATH):
        raise FileNotFoundError(f"❌ ZIP not found at: {ZIP_GDRIVE_PATH}")
    
    zip_size_gb = os.path.getsize(ZIP_GDRIVE_PATH) / (1024**3)
    print(f"📦 Found ZIP: {zip_size_gb:.1f} GB")
    
    LOCAL_ZIP = "/content/yolo_dataset.zip"
    print("📋 Copying ZIP to Colab SSD...")
    t0 = time.time()
    shutil.copy2(ZIP_GDRIVE_PATH, LOCAL_ZIP)
    dt = time.time() - t0
    print(f"✅ Copy done in {dt:.0f}s ({zip_size_gb/dt*1024 if dt > 0 else 0:.0f} MB/s)")
    
    print("📂 Extracting...")
    os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)
    !unzip -qo "{LOCAL_ZIP}" -d "{LOCAL_DATASET_DIR}"
    os.remove(LOCAL_ZIP)
    print("✅ Dataset extracted to SSD.")
else:
    print("✅ Dataset already present.")

from models.hybrid_model import HybridWeaponDetector
print("✅ HybridWeaponDetector ready.")

## Step 3 — Dataset Verification (Sanity Check)
Confirm images/labels count and `data.yaml` integrity.

In [ ]:
import yaml
with open(DATA_YAML_PATH) as f: cfg = yaml.safe_load(f)

print("═" * 50)
print(f"Names : {cfg.get('names')}")
print("═" * 50)

total = 0
for split in ['train', 'val']:
    img_dir = Path(cfg[split])
    if img_dir.exists():
        n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
        total += n_img
        print(f"✅ {split:5s}: {n_img:>6,} images")
    else:
        print(f"⚠️  {split:5s}: directory not found at {img_dir}")

print(f"\n📊 Total images: {total:,}")
if total > 40000: print("✅ Dataset looks complete!")
else: print("⚠️  Image count lower than expected. Check extraction logs.")

## Step 4 — Optimized Hybrid Trainer (Phase-Aware Scheduler)

**V7 Updates**:
- **CosineAnnealingLR**: For smooth LR decay.
- **Differential Phase 2 LR**: LR drops to `1e-5` when unfreezing backbone.
- **Robust Resume**: Correctly reconstructs optimizer param groups when resuming in Phase 2.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from tqdm import tqdm
from pathlib import Path

def get_dataloaders(data_yaml_path: str, batch_size: int = 16, imgsz: int = 640):
    data_cfg = check_det_dataset(data_yaml_path)
    train_set = YOLODataset(
        img_path=data_cfg['train'], imgsz=imgsz,
        augment=True, batch_size=batch_size, task='detect', data=data_cfg
    )
    val_set = YOLODataset(
        img_path=data_cfg['val'], imgsz=imgsz,
        augment=False, batch_size=batch_size, task='detect', data=data_cfg
    )
    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, collate_fn=train_set.collate_fn
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True, collate_fn=val_set.collate_fn
    )
    return train_loader, val_loader

class HybridTrainer:
    def __init__(self, model, train_loader, val_loader, device="cuda", weights_dir="models/weights", freeze_epochs=10):
        self.device = device
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.scaler = GradScaler("cuda")
        self.weights_dir = Path(weights_dir)
        self.freeze_epochs = freeze_epochs
        self.best_loss = float('inf')
        self.criterion = model.head.compute_loss
        self.weights_dir.mkdir(parents=True, exist_ok=True)

    def _set_backbone_frozen(self, frozen: bool):
        for param in self.model.backbone.parameters():
            param.requires_grad = not frozen
        tag = "frozen ❄️" if frozen else "unfrozen 🔥"
        print(f"  Backbone {tag}")

    def _build_optimizer(self, lr=1e-4):
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
        return optimizer

    def run(self, total_epochs=50, resume=True):
        start_epoch = 1
        is_phase2 = False
        
        if resume and (self.weights_dir / "last.pt").exists():
            ckpt = torch.load(self.weights_dir / "last.pt", map_location=self.device)
            start_epoch = ckpt['epoch'] + 1
            is_phase2 = (start_epoch > self.freeze_epochs)
            print(f"✅ Found checkpoint. Resuming from epoch {start_epoch}")
        
        if is_phase2:
            self._set_backbone_frozen(False)
            optimizer = optim.AdamW([
                {'params': self.model.backbone.parameters(), 'lr': 1e-5},
                {'params': self.model.neck.parameters(), 'lr': 5e-5},
                {'params': self.model.head.parameters(), 'lr': 5e-5}
            ], weight_decay=1e-4)
        else:
            self._set_backbone_frozen(True)
            optimizer = self._build_optimizer(lr=1e-4)
            
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)
        
        if resume and (self.weights_dir / "last.pt").exists():
            ckpt = torch.load(self.weights_dir / "last.pt", map_location=self.device)
            self.model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            self.best_loss = ckpt.get('best_loss', float('inf'))
            for _ in range(1, start_epoch): scheduler.step()
        
        for epoch in range(start_epoch, total_epochs + 1):
            if epoch == self.freeze_epochs + 1 and not is_phase2:
                print("\n═══ Phase 2: Unfreezing (Lower LR) ═══")
                self._set_backbone_frozen(False)
                optimizer = optim.AdamW([
                    {'params': self.model.backbone.parameters(), 'lr': 1e-5},
                    {'params': self.model.neck.parameters(), 'lr': 5e-5},
                    {'params': self.model.head.parameters(), 'lr': 5e-5}
                ], weight_decay=1e-4)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs - self.freeze_epochs)
                is_phase2 = True

            self.model.train()
            avg_loss = 0.0
            lr_current = optimizer.param_groups[0]['lr']
            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{total_epochs} [LR={lr_current:.2e}]")
            
            for batch in pbar:
                imgs = batch['img'].to(self.device).float() / 255.0
                optimizer.zero_grad()
                with autocast("cuda"):
                    preds = self.model(imgs)
                    loss  = self.criterion(preds, batch, self.device)
                self.scaler.scale(loss).backward()
                self.scaler.step(optimizer)
                self.scaler.update()
                avg_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            
            avg_loss /= len(self.train_loader)
            
            # ── Validation Phase ──
            self.model.eval()
            val_loss = 0.0
            print(f"  Validating...")
            with torch.no_grad():
                for batch in tqdm(self.val_loader, desc="Validation"):
                    imgs = batch['img'].to(self.device).float() / 255.0
                    with autocast("cuda"):
                        preds = self.model(imgs)
                        loss = self.criterion(preds, batch, self.device)
                    val_loss += loss.item()
            val_loss /= len(self.val_loader)
            
            scheduler.step()
            
            is_best = val_loss < self.best_loss
            if is_best: self.best_loss = val_loss
            
            torch.save({
                'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'best_loss': self.best_loss
            }, self.weights_dir / "last.pt")
            if is_best: torch.save(self.model.state_dict(), self.weights_dir / "best.pt")
            print(f"Epoch {epoch:3d} | Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f} | Best: {self.best_loss:.4f}")


## Step 5 — Launch Optimized Training

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridWeaponDetector(backbone_variant="yolo11m.pt", nc=3, device=device)
model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)

train_loader, val_loader = get_dataloaders(str(DATA_YAML_PATH), batch_size=16)

trainer = HybridTrainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    weights_dir=str(PROJECT_ROOT / "models" / "weights")
)

trainer.run(total_epochs=50, resume=True)

## Step 6 — Visual Verification (Audit)
Run the trained model on a sample image to verify bounding box decoding logic.

In [ ]:
import cv2, glob
import matplotlib.pyplot as plt

# 1. Load a real image from disk (BGR, as OpenCV reads it)
val_img_dir = Path(LOCAL_DATASET_DIR) / "val" / "images"
sample_path = sorted(glob.glob(str(val_img_dir / "*.jpg")))[0]
sample_bgr = cv2.imread(sample_path)

# 2. Run prediction (predict() expects BGR ndarray — matches training)
model.eval()
detections = model.predict(sample_bgr, conf_threshold=0.1)

# 3. Convert to RGB for matplotlib display only
h, w = sample_bgr.shape[:2]
sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 10))
plt.imshow(sample_rgb)
ax = plt.gca()

for det in detections:
    x1 = det['bbox'][0] * w
    y1 = det['bbox'][1] * h
    x2 = det['bbox'][2] * w
    y2 = det['bbox'][3] * h
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red', linewidth=2)
    ax.add_patch(rect)
    ax.text(x1, y1, f"{det['class_name']} {det['confidence']:.2f}",
            bbox=dict(facecolor='red', alpha=0.5), color='white')

plt.axis('off')
plt.title(f"Visual Audit: Found {len(detections)} detections")
plt.show()